In [0]:
from pyspark.sql.functions import col

In [0]:
control_table = "formula1_inc.control.batch_control"

In [0]:
landing_batches = sorted([
    file.name.rstrip('/') 
    for file in dbutils.fs.ls("/Volumes/formula1_inc/landing/files/landing")
    if file.isDir()
])

if spark.catalog.tableExists(control_table):
    tracked_batches = [
                row.batch_id for row in (
                            spark.table(control_table)
                                 .filter(col("status").isin("in_progress","completed"))
                                 .select("batch_id")
                                 .distinct()
                                 .collect()
                     )
    ]
else:
    tracked_batches = []

new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print(f"Landing Branches : {landing_batches}")
print(f"Tracked Branches : {tracked_batches}")
print(f"Next Branch : {next_batch}")

if next_batch is None:
    dbutils.jobs.taskValues.set("p_batch_id","")
    dbutils.jobs.taskValues.set("has_batch","false")
else:
    dbutils.jobs.taskValues.set("p_batch_id",next_batch)
    dbutils.jobs.taskValues.set("has_batch","true")